In [ ]:

import json
from pathlib import Path
from langchain_core.documents import Document

def load_queries(file_path):
    que_doc = []

    file = Path(file_path)

    print(f"\nProcessing {file.name}")

    try:
        with open(file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        # handle single object or list
        if isinstance(raw, dict):
            raw = [raw]

        for idx, entry in enumerate(raw):

            page_content = f"""
            Question: {entry.get('question', '')}
            Intent: {entry.get('intent', '')}
            Business Context: {entry.get('business_context', '')}
            Tables_Used: {entry.get('tables_used', '')}
            embedding_texts: {entry.get('embedding_text', '')}
            SQL Pattern: {entry.get('sql', '')}
            """

            doc = Document(
                page_content=page_content.strip(),
                metadata={
                    "source_file": file.name,
                    "file_type": "json",
                    "index": idx,
                    "question": entry.get("question", ""),
                    "intent": entry.get("intent", ""),
                    "embedding_text": entry.get("embedding_text", ""),
                    "sql": entry.get("sql", ""),
                    "tables_used": entry.get("tables_used", [])
                }
            )

            que_doc.append(doc)

        print(f"Loaded {len(que_doc)} documents from {file.name}")

    except Exception as e:
        print(f"Error loading file {file.name}: {e}")

    return que_doc

query_load=load_queries(r"Queries JSON path")


Processing queries.json
Loaded 13 documents from queries.json


d:\Kathirvelan\Projects\AI_LEARN-PRACTICE\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
#Loading the schema JSON into separate variable
from langchain_core.documents import Document
import json
from pathlib import Path

def load_schema(file_path):
    schema_docs = []

    file = Path(file_path)
    print(f"\nProcessing {file.name}")

    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        for idx, (table_name, table_data) in enumerate(data.items()):

            columns_text = "\n".join([
                f"{col}: {desc}"
                for col, desc in table_data.get("columns", {}).items()
            ])

            joins_text = "\n".join([
                f"{j['table']} ON {j['on']}"
                for j in table_data.get("joins", [])
            ])

            page_content = f"""
            Table: {table_name}

            Description: {table_data.get('description', '')}
            Business Meaning: {table_data.get('business_meaning', '')}

            Embedding Text: {table_data.get('embedding_text', '')}

            Business Terms: {", ".join(table_data.get("business_terms", []))}

            Columns:
            {columns_text}

            Joins:
            {joins_text}
            """

            doc = Document(
                page_content=page_content.strip(),
                metadata={
                    "type": "schema",
                    "table_name": table_name,
                    "source_file": file.name,
                    "embedding_text": table_data.get("embedding_text", "")
                }
            )

            schema_docs.append(doc)

        print(f"Loaded {len(schema_docs)} schema documents")

    except Exception as e:
        print(f"Error loading schema file: {e}")

    return schema_docs
schema_load=load_schema(r"Tables and Columns JSON path")


Processing Tables&Columns.json
Loaded 6 schema documents


In [3]:
from sentence_transformers import SentenceTransformer
import uuid
import chromadb
import numpy as np
from typing import List, Dict, Any
import os

d:\Kathirvelan\Projects\AI_LEARN-PRACTICE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class EmbeddingModel:
    def __init__(self,model_name:str="BAAI/bge-base-en-v1.5"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading Embedding Model {self.model_name}")
            self.model=SentenceTransformer(r"D:/hf_models/bge-base-en-v1.5",local_files_only=True)
            # self.model=SentenceTransformer("BAAI/bge-base-en-v1.5")

            print(f"Model Loaded. Embeddings Dimension {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name} : {e}")

    def generate_embeddings(self,file:List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model Not Found")
        print(f"Generating Embeddings for total {len(file)} documents")
        embeddings=self.model.encode(file,show_progress_bar=True)
        print(f"Completed embeddings for size of {embeddings.shape}")
        return embeddings


In [5]:
embedding_manager=EmbeddingModel()
embedding_manager

Loading Embedding Model BAAI/bge-base-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6027.95it/s]

Model Loaded. Embeddings Dimension 768



C:\Users\Techative\AppData\Local\Temp\ipykernel_27416\2950896471.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded. Embeddings Dimension {self.model.get_sentence_embedding_dimension()}")


In [6]:
class Schema_VectorDB:
    def __init__(self, sche_collection_name="Schema",
                 # ✅ THIS LINE CHANGES
                 persist_directory: str = r"D:\Kathirvelan\Projects\AI_LEARN-PRACTICE\data\schema_coll"):
        self.collection_name = sche_collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"Description": "Schema details Embeddings"}
            )
            print(f"Vector Embeddings initialized in the name of {self.collection_name}")
            print(f"Total documents in the collection is {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing the vector DB {e}")
            raise

    def add_docs(self, documents: List[Any], embeddings: np.ndarray):
        if self.collection.count() > 0:
            print(f"Already has {self.collection.count()} docs — skipping indexing")
            return
        ids, metadatas, document_texts, embedding_lists = [], [], [], []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadatas.append(metadata)
            document_texts.append(doc.page_content)
            embedding_lists.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids, embeddings=embedding_lists,
                metadatas=metadatas, documents=document_texts,
            )
            print(f"Successfully added {len(documents)} docs. Total: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding embeddings: {e}")
            raise

sche_db = Schema_VectorDB()

Vector Embeddings initialized in the name of Schema
Total documents in the collection is 6


In [7]:
class Query_VectorDB:
    def __init__(self, collection_name: str = "Queries",
                 # ✅ THIS LINE CHANGES
                 persist_directory: str = r"D:\Kathirvelan\Projects\AI_LEARN-PRACTICE\data\queries_coll"):
        self.collection_name = collection_name
        self.presist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.presist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.presist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"Description": "Example Queries Embeddings"}
            )
            print(f"Vector Embeddings initialized in the name of {self.collection_name}")
            print(f"Total documents in the collection is {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing the vector DB {e}")
            raise

    def add_docs(self, documents: List[Any], embeddings: np.ndarray):
        if self.collection.count() > 0:
            print(f"Already has {self.collection.count()} docs — skipping indexing")
            return
        ids, metadatas, document_texts, embedding_lists = [], [], [], []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadatas.append(metadata)
            document_texts.append(doc.page_content)
            embedding_lists.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids, embeddings=embedding_lists,
                metadatas=metadatas, documents=document_texts,
            )
            print(f"Successfully added {len(documents)} docs. Total: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding embeddings: {e}")
            raise

que_db = Query_VectorDB()

Vector Embeddings initialized in the name of Queries
Total documents in the collection is 13


In [8]:
# ✅ Run this ONLY on first time
# From second session onwards — SKIP THIS CELL, data is already in Drive

if que_db.collection.count() == 0:
    que_texts = [doc.metadata["embedding_text"] for doc in query_load]
    que_embeddings = embedding_manager.generate_embeddings(que_texts)
    que_db.add_docs(query_load, que_embeddings)
else:
    print(f"Queries already indexed: {que_db.collection.count()} docs")

if sche_db.collection.count() == 0:
    sche_texts = [doc.metadata["embedding_text"] for doc in schema_load]
    sche_embeddings = embedding_manager.generate_embeddings(sche_texts)
    sche_db.add_docs(schema_load, sche_embeddings)
else:
    print(f"Schema already indexed: {sche_db.collection.count()} docs")

Queries already indexed: 13 docs
Schema already indexed: 6 docs


In [15]:
class RAGRetriever:
    def __init__(self, queries_store, schema_store, embedding_manager):
        self.queries_store = queries_store
        self.schema_store = schema_store
        self.embedding_manager = embedding_manager

    def _search(self, collection, question_embedding, top_k):
        result = collection.query(
            query_embeddings=[question_embedding.tolist()],
            n_results=top_k
        )
        docs = []
        if result['documents'] and result['documents'][0]:
            for i, (doc_id, document, metadata, distance) in enumerate(zip(
                result['ids'][0], result['documents'][0],
                result['metadatas'][0], result['distances'][0],
            )):
                docs.append({
                    "id": doc_id, "content": document,
                    "metadata": metadata, "distance": distance,
                    "score": 1 - distance ,   # ✅ fixed score formula
                    "rank": i + 1
                })
        return docs

    def retrieve(self, question: str, top_k_schemas=2, top_k_queries=2, score_threshold=0):
        print("\nThe question is :", question)
        question_embedding = self.embedding_manager.generate_embeddings([question])[0]
        schema_results  = self._search(self.schema_store.collection,  question_embedding, top_k_schemas)
        queries_results = self._search(self.queries_store.collection, question_embedding, top_k_queries)
        schema_results  = [d for d in schema_results  if d["score"] >= score_threshold]
        queries_results = [d for d in queries_results if d["score"] >= score_threshold]
        print(f"Found totally {len(schema_results)} schema matches")
        print(f"Found totally {len(queries_results)} queries matches")
        return {"schema": schema_results, "queries": queries_results}

import requests

class Ollama:
    def __init__(self, model_name="gemma3:4b", url="http://localhost:11434"):
        self.model = model_name
        self.url = f"{url}/api/generate"

    def invoke(self, query: str):
        response = requests.post(
            self.url,
            json={"model": self.model, "prompt": query, "stream": False}
        )
        result = response.json()

        # ✅ Add this to see what's actually coming back
        if "response" not in result:
            print("❌ Unexpected response from Ollama:")
            print(result)
            return result.get("error", "Unknown error from Ollama")

        return result["response"]

# llm = Ollama()

HANA_RULES = """
STRICT RULES:
- Return ONLY the SQL query — no explanation, no markdown, no ``` ```
- Don't use DATE function before dates
- Use ONLY tables and columns from SCHEMA_DETAILS context
- Always alias tables: FROM OINV T0, JOIN OCRD T1
- Use table alias name while using corresponding column like T0."ItemCode"
- ALWAYS wrap EVERY column name in "" like T0."DocEntry" WITHOUT EXCEPTION
- Use COUNT(*) for counting, LIMIT N for row limiting
- Use IFNULL() not ISNULL()
"""

def prompt_build(query, schema_context, queries_context):
    return f"""You are a SAP HANA SQL expert.
{HANA_RULES}
SCHEMA_DETAILS:
{schema_context if schema_context else "No schema details found"}
QUERY_EXAMPLES:
{queries_context if queries_context else "No similar examples found"}
USER_QUERY: {query}
SQL:"""

def final_rag(query, retriever, llm, top_k=2, score_threshold=0.2):
    results = retriever.retrieve(query)
    schema_results  = results["schema"][:top_k]
    queries_results = results["queries"][:top_k]
    if not schema_results:
        print("⚠️ Warning: No schema matches found")
    if not queries_results:
        print("⚠️ Warning: No query examples found")
    schema_context  = "\n\n--\n\n".join([doc["content"] for doc in schema_results])
    queries_context = "\n\n--\n\n".join([doc["content"] for doc in queries_results])
    prompt = prompt_build(query, schema_context, queries_context)
    print("Prompt length:", len(prompt))
    return llm.invoke(prompt)

# Initialize
llm = Ollama()
rag_retriever = RAGRetriever(que_db, sche_db, embedding_manager)
print("✅ Ready")

✅ Ready


In [16]:
Question="top most selling 5 items based on quantity"
ans = final_rag(Question, rag_retriever, llm)
print(ans)


The question is : top most selling 5 items based on quantity
Generating Embeddings for total 1 documents


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.24it/s]

Completed embeddings for size of (1, 768)
Found totally 2 schema matches
Found totally 2 queries matches
Prompt length: 2818


SELECT T1."ItemCode", SUM(T0."Quantity") AS "TotalQuantity" FROM INV1 T0 JOIN OITM T1 ON T0."ItemCode" = T1."ItemCode" GROUP BY T1."ItemCode" ORDER BY "TotalQuantity" DESC LIMIT 5



In [11]:
rag_retriever.retrieve(Question)


The question is : top most selling 5 items
Generating Embeddings for total 1 documents


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.34it/s]

Completed embeddings for size of (1, 768)
Found totally 2 schema matches
Found totally 2 queries matches


{'schema': [{'id': '822a5a70_3',
   'content': 'Table: INV1\n\n            Description: Invoice line items\n            Business Meaning: item level sales data, quantity sold per product\n\n            Embedding Text: sold items sales quantity product revenue item sales invoice line items sold quantity per product\n\n            Business Terms: sales items, sold quantity\n\n            Columns:\n            DocEntry: link to invoice\nItemCode: product id\nQuantity: sold quantity\nLineTotal: sales revenue\n\n            Joins:\n            OINV ON INV1.DocEntry = OINV.DocEntry\nOITM ON INV1.ItemCode = OITM.ItemCode',
   'metadata': {'table_name': 'INV1',
    'embedding_text': 'sold items sales quantity product revenue item sales invoice line items sold quantity per product',
    'doc_index': 3,
    'type': 'schema',
    'source_file': 'Tables&Columns.json'},
   'distance': 0.6319031119346619,
   'score': 0.36809688806533813,
   'rank': 1},
  {'id': '5438e0a1_1',
   'content': 'Table: OI

In [12]:
import re

def extract_query(text:str)->str:
    text=re.sub(r"```.*?```", "",text,flags=re.DOTALL)

    match=re.search(r"(SELECT|WITH)\s+.*",text,re.IGNORECASE | re.DOTALL)

    if match:
        sql=match.group(0)
    else:
        return ""
    
    sql=sql.split(";")[0]+";"
    return sql.strip()

cleaned_query=extract_query(ans)
print(cleaned_query)

SELECT T1."ItemName", COUNT(*) AS "SalesCount" FROM INV1 T0 JOIN OITM T1 ON T0."ItemCode" = T1."ItemCode" GROUP BY T1."ItemName" ORDER BY "SalesCount" DESC LIMIT 5;


In [13]:
def run_query(query:str)->str:
    url="http://vzone.in:1662/api/GetMethod/GetData"
    response=requests.get(
        url,
        params={"query": query},
   
    )
    result=response.json()
    return result

final=run_query(cleaned_query)
print(final)
print(Question)

{'status': 403, 'message': 'Forbidden'}
top most selling 5 items


In [14]:
prompt = f"""
You are a data analyst generating a concise business explanation.

Context:
- User Question: {Question}
- SQL Result: {final}

Instructions:
- Provide a clear, precise explanation of the result.
- Use a professional and neutral tone.
- Do NOT restate the question.
- Do NOT add assumptions or external information beyond the result.
- If the result is numeric, interpret it meaningfully in one sentence.
- After answering ask any 2 recommendation questions

Output:
- Return ONLY the final explanation as a short paragraph (max 2–3 sentences).
"""

last=llm.invoke(prompt)
print(last)

The result indicates a '403 Forbidden' error status, signifying that the user lacks the necessary permissions to access the requested data, specifically the top five most selling items. This implies a possible restriction on querying or viewing sales reports directly from the database. 

To proceed, I recommend:
1. Verify if your user account has the appropriate permissions for retrieving sales data.
2. Confirm with the system administrator about correct access procedures and potential workarounds for obtaining this information.
